# RansKnow Dataset — Getting Started

**RansKnow** is a curated collection of **718 ransomware-related video transcripts** from 50 cybersecurity YouTube channels, annotated with MITRE ATT&CK tactic features and offensive tool mentions.

This notebook covers:
1. Exploring the feature CSV (718 rows × 37 columns)
2. Reading a sample transcript
3. Tactic and tool distribution plots
4. Filtering by ransomware family

In [ ]:
import os, zipfile
import pandas as pd
import matplotlib.pyplot as plt

BASE = '/kaggle/input/datasets/henrykabuye/ransknow-v1'
print('Files in dataset:')
for f in sorted(os.listdir(BASE)):
    size = os.path.getsize(f'{BASE}/{f}') if os.path.isfile(f'{BASE}/{f}') else '<dir>'
    print(f'  {f}  ({size})')

In [ ]:
df = pd.read_csv(f'{BASE}/Knowledge_Agent_Features_718.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

## Dataset Overview

In [ ]:
ch_col   = next((c for c in df.columns if 'channel' in c.lower() and 'name' in c.lower()), df.columns[1])
year_col = next((c for c in df.columns if 'year' in c.lower()), None)

print(f'Unique channels: {df[ch_col].nunique()}')
print(f'Total videos:    {len(df)}')
if year_col:
    print(f'Year range:      {int(df[year_col].min())} – {int(df[year_col].max())}')
display(df[[ch_col, year_col]].value_counts().head(10).reset_index() if year_col else df[ch_col].value_counts().head(10))

## Sample Transcript

Each transcript `.txt` file uses `M:SS` timestamps. Transcripts are in the `transcripts/` subdirectory, organised by channel.

In [ ]:
transcript_text = None
transcript_name = None

# Strategy 1: transcripts is a directory (Kaggle v2 dataset)
tx_dir = f'{BASE}/transcripts'
if os.path.isdir(tx_dir):
    for channel_dir in sorted(os.listdir(tx_dir)):
        ch_path = f'{tx_dir}/{channel_dir}'
        if not os.path.isdir(ch_path):
            continue
        for fname in sorted(os.listdir(ch_path)):
            if fname.endswith('.txt'):
                fpath = f'{ch_path}/{fname}'
                raw = open(fpath, encoding='utf-8', errors='replace').read().strip()
                if raw and raw != '[NO TRANSCRIPT AVAILABLE]':
                    transcript_text = raw
                    transcript_name = f'{channel_dir}/{fname}'
                    break
        if transcript_text:
            break

# Strategy 2: transcripts is a zip file
if not transcript_text:
    zips = [f for f in os.listdir(BASE) if f.endswith('.zip') and 'transcript' in f.lower()]
    if not zips:
        zips = [f for f in os.listdir(BASE) if f.endswith('.zip')]
    if zips:
        with zipfile.ZipFile(f'{BASE}/{zips[0]}') as z:
            txts = sorted(n for n in z.namelist() if n.endswith('.txt') and '__MACOSX' not in n)
            for name in txts:
                raw = z.read(name).decode('utf-8', errors='replace').strip()
                if raw and raw != '[NO TRANSCRIPT AVAILABLE]':
                    transcript_text = raw
                    transcript_name = name
                    break

if transcript_text:
    print(f'\n--- {transcript_name} (first 25 lines) ---')
    print('\n'.join(transcript_text.split('\n')[:25]))
else:
    print('No transcript found. Files at BASE:', os.listdir(BASE))

## MITRE ATT&CK Tactic Distribution

In [ ]:
tactic_cols = [c for c in df.columns
               if c.startswith('Tactic_') and c not in ('Tactic_Total_Mentions','Dominant_Tactic')]
if tactic_cols:
    tactic_sums = df[tactic_cols].sum().sort_values()
    labels = [c.replace('Tactic_','').replace('_',' ') for c in tactic_sums.index]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(labels, tactic_sums.values, color='steelblue')
    ax.set_xlabel('Videos mentioning tactic')
    ax.set_title('MITRE ATT&CK Tactic Coverage — RansKnow')
    plt.tight_layout()
    plt.show()
else:
    print('No Tactic_ columns found. Columns:', df.columns.tolist())

## Top Offensive Tools

In [ ]:
tool_cols = [c for c in df.columns
             if c.startswith('Tool_') and c not in ('Tool_Total_Mentions','Tool_List')]
if tool_cols:
    tool_sums = df[tool_cols].sum().sort_values(ascending=False)
    labels = [c.replace('Tool_','').replace('_',' ') for c in tool_sums.index]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, tool_sums.values, color='tomato')
    ax.set_ylabel('Videos mentioning tool')
    ax.set_title('Top Offensive Tools — RansKnow')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## Filter by Ransomware Family

In [ ]:
family_col = next((c for c in df.columns if 'family' in c.lower() and 'list' in c.lower()), None)
title_col  = next((c for c in df.columns if 'title' in c.lower()), None)
vid_col    = next((c for c in df.columns if c.lower() == 'video_id'), df.columns[0])

if family_col:
    lockbit = df[df[family_col].str.contains('lockbit', case=False, na=False)]
    print(f'Videos mentioning LockBit: {len(lockbit)}')
    cols = [c for c in [vid_col, ch_col, title_col, year_col] if c]
    display(lockbit[cols].reset_index(drop=True))
else:
    print('Family_List column not found. Available columns:', df.columns.tolist())

## Videos per Channel

In [ ]:
counts = df.groupby(ch_col).size().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(counts.index[::-1], counts.values[::-1], color='mediumseagreen')
ax.set_xlabel('Videos')
ax.set_title('Top 20 Channels by Video Count — RansKnow')
plt.tight_layout()
plt.show()